# 第Ⅱ部 6章 Agent Platform Feature Storeによる特徴量管理

本ノートブックは、第Ⅱ部 6章のハンズオン用サンプルコードです。
Colab Enterprise 上で、上のセルから順に実行してください。

実行後は課金を避けるため、末尾の「6.14 クリーンアップ」セル （または `clean_up.py`）を必ず実行してください。

## 6.4 環境構築

必要なパッケージをインストールします。

In [ ]:
# パッケージのインストール
# protobuf は 4.x 系を指定（5.x では MessageFactory.GetPrototype が削除され、
# google-cloud-aiplatform の一部モジュールで AttributeError が発生するため）
!pip install --upgrade --quiet \
    "google-cloud-aiplatform>=1.60.0" \
    "google-cloud-bigquery>=3.20.0" \
    "db-dtypes>=1.2.0" \
    "protobuf>=4.25.0,<5.0.0"

### プロジェクト設定

`PROJECT_ID` にご自身のGoogle CloudプロジェクトIDを入力してください。

In [ ]:
PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-central1"
BQ_DATASET_LOCATION = "US"

### SDKの初期化

In [ ]:
import vertexai
from google.cloud import bigquery

vertexai.init(project=PROJECT_ID, location=LOCATION)
bq_client = bigquery.Client(project=PROJECT_ID)

## 6.5 特徴量の設計

ハンズオンで使用するデータソースとリソース名の定数を定義します。

In [ ]:
import time

# --- データソース ---
SOURCE_TABLE_2021 = (
    "bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2021"
)
SOURCE_TABLE_2022 = (
    "bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022"
)

# --- リソース名の定数 ---
# 特徴量テーブルを格納するBigQueryデータセット名
BQ_DATASET_ID = "features"
# 特徴量テーブル名
BQ_TABLE_ID = "zone_stats"
# オンラインサービング基盤のリソース名
FEATURE_ONLINE_STORE_ID = "taxi_online_store"
# FeatureGroup名
FEATURE_GROUP_ID = "zone_stats"
# FeatureView名
FEATURE_VIEW_REGISTRY_ID = "zone_stats_registry_view"
# 登録する特徴量カラム
FEATURE_IDS = ["trip_count", "avg_fare", "avg_distance", "tip_rate"]
# 1/10 ≈ 10% サンプリング
SAMPLE_MOD = 10

# BigQueryテーブルを指すURI。Feature Store APIがデータソース参照時にこの形式を使用
BQ_SOURCE_URI = f"bq://{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}"

### 集計クエリの定義

In [ ]:
_SAMPLE_COND = f"MOD(ABS(FARM_FINGERPRINT(CAST(pickup_datetime AS STRING))), {SAMPLE_MOD}) = 0"

_BASE_WHERE = f"""
trip_distance > 0
AND fare_amount > 0
AND pickup_location_id IS NOT NULL
AND {_SAMPLE_COND}
"""

FEATURE_QUERY = f"""
WITH src AS (
  SELECT pickup_location_id, fare_amount, tolls_amount,
         trip_distance, tip_amount, snapshot
  FROM `{SOURCE_TABLE_2021}`,
       UNNEST([
         TIMESTAMP('2021-07-01'),
         TIMESTAMP('2022-01-01'),
         TIMESTAMP('2022-07-01'),
         TIMESTAMP('2023-01-01')
       ]) AS snapshot
  WHERE {_BASE_WHERE}
    AND pickup_datetime >= '2021-01-01'
    AND pickup_datetime <  '2022-01-01'
    AND pickup_datetime <  snapshot
  UNION ALL
  SELECT pickup_location_id, fare_amount, tolls_amount,
         trip_distance, tip_amount, snapshot
  FROM `{SOURCE_TABLE_2022}`,
       UNNEST([
         TIMESTAMP('2022-07-01'),
         TIMESTAMP('2023-01-01')
       ]) AS snapshot
  WHERE {_BASE_WHERE}
    AND pickup_datetime >= '2022-01-01'
    AND pickup_datetime <  '2023-01-01'
    AND pickup_datetime <  snapshot
)
SELECT
  pickup_location_id AS entity_id,
  COUNT(*) AS trip_count,
  CAST(AVG(fare_amount + tolls_amount) AS FLOAT64) AS avg_fare,
  CAST(AVG(trip_distance) AS FLOAT64) AS avg_distance,
  CAST(SAFE_DIVIDE(SUM(tip_amount), SUM(fare_amount)) AS FLOAT64) AS tip_rate,
  snapshot AS feature_timestamp
FROM src
GROUP BY entity_id, feature_timestamp
HAVING trip_count >= 10
"""

### 集計結果のプレビュー

In [ ]:
df_preview = bq_client.query(
    FEATURE_QUERY, location=BQ_DATASET_LOCATION
).to_dataframe()

# 同一entity_idで年ごとの差を確認
print("\n例: entity_id='237' の年別比較")
df_preview[df_preview["entity_id"] == "237"].sort_values("feature_timestamp")

### BigQueryテーブルの作成

In [ ]:
dataset_ref = f"{PROJECT_ID}.{BQ_DATASET_ID}"
dataset = bigquery.Dataset(dataset_ref)
dataset.location = BQ_DATASET_LOCATION
dataset = bq_client.create_dataset(dataset, exists_ok=True)
print(f"データセット作成完了: {dataset_ref} (location={BQ_DATASET_LOCATION})")

In [ ]:
table_ref = f"{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}"
create_table_sql = f"CREATE OR REPLACE TABLE `{table_ref}` AS {FEATURE_QUERY}"

bq_client.query(create_table_sql, location=BQ_DATASET_LOCATION).result()

# 作成結果を確認
row_count = bq_client.query(
    f"SELECT COUNT(*) AS cnt FROM `{table_ref}`", location=BQ_DATASET_LOCATION
).to_dataframe().iloc[0]["cnt"]
print(f"テーブル作成完了: {table_ref} ({row_count} 行)")

## 6.6 FeatureOnlineStoreの作成

In [ ]:
from vertexai.resources.preview import feature_store

fos = feature_store.FeatureOnlineStore.create_bigtable_store(
    FEATURE_ONLINE_STORE_ID
)
print(f"FeatureOnlineStore作成完了: {fos.name}")

## 6.7 FeatureViewの作成

### FeatureGroupの作成

In [ ]:
fg = feature_store.FeatureGroup.create(
    name=FEATURE_GROUP_ID,
    source=feature_store.utils.FeatureGroupBigQuerySource(
        uri=BQ_SOURCE_URI,
        entity_id_columns=["entity_id"],
    ),
)
print(f"FeatureGroup作成完了: {fg.name}")

### Featureの登録

In [ ]:
feature_configs = [
    {"name": "trip_count", "description": "ゾーンのトリップ総数"},
    {"name": "avg_fare", "description": "平均運賃"},
    {"name": "avg_distance", "description": "平均走行距離"},
    {"name": "tip_rate", "description": "チップ率"},
]

for config in feature_configs:
    feat = fg.create_feature(
        name=config["name"], description=config["description"]
    )
    print(f"  Feature作成完了: {config['name']}")

print(f"\n全 {len(feature_configs)} 個のFeatureを登録しました")

### Feature Registryを経由したFeatureViewの作成

In [ ]:
from google.cloud.aiplatform_v1 import FeatureOnlineStoreAdminServiceClient
from google.cloud.aiplatform_v1.types import (
    feature_online_store_admin_service as fos_admin_pb2,
    feature_view as feature_view_pb2,
)

admin_client = FeatureOnlineStoreAdminServiceClient(
    client_options={"api_endpoint": f"{LOCATION}-aiplatform.googleapis.com"}
)

feature_registry_source = feature_view_pb2.FeatureView.FeatureRegistrySource(
    feature_groups=[
        feature_view_pb2.FeatureView.FeatureRegistrySource.FeatureGroup(
            feature_group_id=FEATURE_GROUP_ID,
            feature_ids=FEATURE_IDS,
        )
    ]
)

create_view_lro = admin_client.create_feature_view(
    fos_admin_pb2.CreateFeatureViewRequest(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}"
               f"/featureOnlineStores/{FEATURE_ONLINE_STORE_ID}",
        feature_view_id=FEATURE_VIEW_REGISTRY_ID,
        feature_view=feature_view_pb2.FeatureView(
            feature_registry_source=feature_registry_source,
        ),
    )
)

# Long-Running Operationの完了を待機
fv_registry = create_view_lro.result()
print(f"FeatureView作成完了: {fv_registry.name}")

## 6.8 データ同期

> **トラブルシューティング**: 同期が完了と表示されても 6.12 のオンライン特徴量取得で値が取得できない（`404 Not Found` になる）場合は、コンソール（Agent Platform Feature Store → FeatureView → 同期）で同期ジョブのステータスを確認してください。  
> 権限エラーで失敗している場合は、Vertex AI Service Agent（`service-プロジェクト番号@gcp-sa-aiplatform.iam.gserviceaccount.com`）に Bigtable への書き込み権限（`roles/bigtable.user` など）を付与してから、同期を再実行してください。

In [ ]:
# 手動Syncのトリガー
fv_registry = feature_store.FeatureView(
    FEATURE_VIEW_REGISTRY_ID,
    feature_online_store_id=FEATURE_ONLINE_STORE_ID,
)
sync = fv_registry.sync()
print(f"Syncを開始しました: {sync.resource_name}")
sync.wait()
print("Syncが正常に完了しました")

## 6.9 定期的なデータ同期の設定

`sync_config` を変更することで、スケジュール同期／継続的同期のFeatureViewを作成できます。

In [ ]:
# スケジュール同期の設定例
sync_config=feature_view_pb2.FeatureView.SyncConfig(
    cron="0 3 * * *",
)

# 継続的同期の設定例
sync_config=feature_view_pb2.FeatureView.SyncConfig(
    continuous=True,
)

### 継続的同期のシミュレーション（任意）

新しいレコード（`entity_id = "9999"`）が数分以内にオンラインストアへ自動反映されることを確認します。

In [ ]:
# 継続的同期のシミュレーション（任意）
#
# continuous=True のFeatureViewを作成し、BigQueryソーステーブルへの
# INSERTが数分以内にオンラインストアへ自動反映されることを確認します。
# 実行には数分かかるため、必要な場合のみ True にしてください。
RUN_CONTINUOUS_SYNC_DEMO = False  # @param {type:"boolean"}

CONTINUOUS_FEATURE_VIEW_ID = "zone_stats_continuous_view"
DEMO_ENTITY_ID = "9999"
MAX_WAIT_SECONDS = 600

if RUN_CONTINUOUS_SYNC_DEMO:
    # 1. 継続的同期のFeatureViewを作成する
    continuous_lro = admin_client.create_feature_view(
        fos_admin_pb2.CreateFeatureViewRequest(
            parent=f"projects/{PROJECT_ID}/locations/{LOCATION}"
                   f"/featureOnlineStores/{FEATURE_ONLINE_STORE_ID}",
            feature_view_id=CONTINUOUS_FEATURE_VIEW_ID,
            feature_view=feature_view_pb2.FeatureView(
                feature_registry_source=feature_registry_source,
                sync_config=feature_view_pb2.FeatureView.SyncConfig(
                    continuous=True,
                ),
            ),
        )
    )
    print(f"継続的同期のFeatureView作成完了: {continuous_lro.result().name}")

    # 2. BigQueryのソーステーブルに新しいレコードを1件INSERTする
    insert_sql = f"""
    INSERT INTO `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
      (entity_id, trip_count, avg_fare, avg_distance, tip_rate, feature_timestamp)
    VALUES
      ('{DEMO_ENTITY_ID}', 100, 12.34, 2.5, 0.18, CURRENT_TIMESTAMP())
    """
    bq_client.query(insert_sql, location=BQ_DATASET_LOCATION).result()
    print(f"INSERT完了: entity_id='{DEMO_ENTITY_ID}'")

    # 3. オンラインストアへの自動反映をポーリングで待機する
    fv_continuous = feature_store.FeatureView(
        CONTINUOUS_FEATURE_VIEW_ID,
        feature_online_store_id=FEATURE_ONLINE_STORE_ID,
    )
    start = time.time()
    while time.time() - start < MAX_WAIT_SECONDS:
        try:
            demo_result = fv_continuous.read(key=[DEMO_ENTITY_ID])
            # 未反映の場合はレスポンスに特徴量が含まれない
            if "feature_timestamp" in str(demo_result):
                print(f"\n自動反映を確認しました（{int(time.time() - start)}秒）")
                print(demo_result)
                break
        except Exception:
            pass
        print("  反映を待機中...")
        time.sleep(30)
    else:
        print("タイムアウトしました。数分後にもう一度実行して確認してください")

    # 作成したFeatureViewは6.14のクリーンアップでまとめて削除されます
else:
    print("RUN_CONTINUOUS_SYNC_DEMO = False のため、シミュレーションは実行されません")


## 6.11 学習時の特徴量取得（PIT取得）

In [ ]:
learning_data_query = f"""
SELECT
  t.pickup_location_id,
  t.pickup_datetime,
  z.feature_timestamp,        -- PITで選択された特徴量バージョン
  -- リクエストに含まれる特徴量（トリップテーブルから）
  CAST(t.trip_distance AS FLOAT64) AS trip_distance,
  t.passenger_count,
  EXTRACT(HOUR FROM t.pickup_datetime) AS hourofday,
  -- Feature Storeの集計特徴量（BigQueryテーブルからPIT取得）
  z.trip_count   AS zone_trip_count,
  z.avg_fare     AS zone_avg_fare,
  z.avg_distance AS zone_avg_distance,
  z.tip_rate     AS zone_tip_rate,
  -- 予測対象
  CAST(t.tip_amount AS FLOAT64) AS label
FROM `{SOURCE_TABLE_2022}` AS t
LEFT JOIN `{PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}` AS z
  ON t.pickup_location_id = z.entity_id
  AND z.feature_timestamp <= t.pickup_datetime  -- 乗車時点以前の特徴量に限定
WHERE
  t.trip_distance > 0
  AND t.fare_amount > 0
  AND t.tip_amount > 0
  AND t.passenger_count IS NOT NULL
  AND (
    -- 2022年前半（3月）と後半（9月）を混在させてPITの動作を確認
    (t.pickup_datetime >= '2022-03-01' AND t.pickup_datetime < '2022-04-01')
    OR
    (t.pickup_datetime >= '2022-09-01' AND t.pickup_datetime < '2022-10-01')
  )
QUALIFY ROW_NUMBER() OVER (   -- 条件を満たす複数バージョンのうち最新を選択
  PARTITION BY t.pickup_location_id, t.pickup_datetime
  ORDER BY z.feature_timestamp DESC
) = 1
ORDER BY t.pickup_datetime
"""

df_learning = bq_client.query(
    learning_data_query, location=BQ_DATASET_LOCATION
).to_dataframe()
print(f"学習データ ({len(df_learning)} 行)")

# 同じゾーンで3月と9月のfeature_timestampの違いを確認
sample_zone = df_learning["pickup_location_id"].iloc[0]
print(f"\n例: pickup_location_id='{sample_zone}' のトリップ（PITの確認）")
df_learning[df_learning["pickup_location_id"] == sample_zone].sort_values("pickup_datetime")

## 6.12 推論時のオンラインサービング

> **補足**: 取得される特徴量の数値（`trip_count` など）は、書籍掲載の出力例とわずかに異なる場合があります（公開データセットの更新などによるもので、動作上の問題はありません）。  
> また、同期完了直後は反映まで数十秒かかることがあります。値が取得できない場合は少し待ってから再実行してください。

In [ ]:
# 既存を取得（createせずに名前で参照）
fg = feature_store.FeatureGroup("zone_stats")
# 既存のFeatureViewを名前で参照（createではなくコンストラクタで取得）
fv = feature_store.FeatureView(
    FEATURE_VIEW_REGISTRY_ID,
    feature_online_store_id=FEATURE_ONLINE_STORE_ID,
)

# entity_idを指定して特徴量を取得
result = fv.read(key=["237"])
print("entity_id = 237の特徴量:")
print(result)

## 6.13 Feature Monitoringによるドリフト検出

### FeatureMonitorの作成

In [ ]:
FEATURE_MONITOR_ID = "zone_stats_monitor"
DRIFT_THRESHOLD = 0.3

# 監視対象のFeatureとドリフトしきい値の設定
feature_selection_configs = [
    (feature_id, DRIFT_THRESHOLD) for feature_id in FEATURE_IDS
]

# FeatureMonitorの作成
feature_monitor = fg.create_feature_monitor(
    name=FEATURE_MONITOR_ID,
    feature_selection_configs=feature_selection_configs,
    schedule_config="0 */6 * * *",
)
print(f"FeatureMonitor作成完了: {feature_monitor.name}")

### Feature Monitoring Jobの実行（ベースライン取得）

In [ ]:
import time

# 1回目のFeature Monitoring Jobを手動実行
job_1 = feature_monitor.create_feature_monitor_job()
job_id = job_1.name.split("/")[-1]
print(f"Job作成完了: {job_id}")

# Jobの完了をポーリングで待機
while True:
    job_1 = feature_monitor.get_feature_monitor_job(job_id)
    try:
        if job_1.feature_stats_and_anomalies:
            print("Job完了")
            break
    except Exception:
        pass
    print("  実行中...")
    time.sleep(30)

### ベースラインの結果確認

In [ ]:
for stat in job_1.feature_stats_and_anomalies:
    print(f"Feature: {stat.feature_id}")
    print(f"  distribution_deviation: {stat.distribution_deviation}")
    print(f"  drift_detected:         {stat.drift_detected}")

### ドリフトのシミュレーションと結果の可視化

意図的に分布を変化させたデータを挿入し、2回目のFeature Monitoring Jobでドリフトが検出されることを確認します。

In [ ]:
# ドリフトのシミュレーション
# 分布が大きく偏ったデータ（運賃を増額し、チップ率を低下させたもの）を
# 新しいfeature_timestampとしてBigQueryテーブルに挿入する
drift_sql = f"""
INSERT INTO `{table_ref}`
  (entity_id, trip_count, avg_fare, avg_distance, tip_rate, feature_timestamp)
SELECT
  entity_id,
  CAST(trip_count * 3 AS INT64) AS trip_count,
  avg_fare * 2.5      AS avg_fare,       -- 運賃を大幅に増額
  avg_distance * 2.0  AS avg_distance,   -- 走行距離を倍増
  tip_rate * 0.3      AS tip_rate,       -- チップ率を大幅に低下
  TIMESTAMP('2023-07-01') AS feature_timestamp
FROM `{table_ref}`
WHERE feature_timestamp = TIMESTAMP('2023-01-01')
"""

bq_client.query(drift_sql, location=BQ_DATASET_LOCATION).result()

drift_row_count = bq_client.query(
    f"SELECT COUNT(*) AS cnt FROM `{table_ref}`", location=BQ_DATASET_LOCATION
).to_dataframe().iloc[0]["cnt"]
print(f"ドリフト用データを挿入しました（テーブル全体: {drift_row_count} 行）")

# オンラインストアのスナップショットも最新化しておく
sync_2 = fv.sync()
sync_2.wait()
print("Syncが正常に完了しました")


In [ ]:
# 2回目のFeature Monitoring Jobを手動実行し、1回目との分布を比較する
job_2 = feature_monitor.create_feature_monitor_job()
job_2_id = job_2.name.split("/")[-1]
print(f"Job作成完了: {job_2_id}")

while True:
    job_2 = feature_monitor.get_feature_monitor_job(job_2_id)
    try:
        if job_2.feature_stats_and_anomalies:
            print("Job完了\n")
            break
    except Exception:
        pass
    print("  実行中...")
    time.sleep(30)

for stat in job_2.feature_stats_and_anomalies:
    print(f"Feature: {stat.feature_id}")
    print(f"  distribution_deviation: {stat.distribution_deviation}")
    print(f"  drift_detected:         {stat.drift_detected}")


In [ ]:
# 過去のFeature Monitoring Jobの履歴を取得し、ドリフトの推移を可視化する
import matplotlib.pyplot as plt

jobs = list(feature_monitor.list_feature_monitor_jobs())


def _created_at(job):
    """作成時刻（取得できない場合はJob ID）でソートするためのキー"""
    created = getattr(job, "create_time", None)
    return str(created) if created is not None else job.name


jobs = sorted(jobs, key=_created_at)

# 特徴量ごとにdistribution_deviationの推移を集計する
history = {feature_id: [] for feature_id in FEATURE_IDS}
labels = []
latest_stats = {}

for job in jobs:
    stats = list(getattr(job, "feature_stats_and_anomalies", []) or [])
    if not stats:
        continue
    labels.append(f"Job {len(labels) + 1}")
    stats_by_id = {stat.feature_id: stat for stat in stats}
    for feature_id in FEATURE_IDS:
        stat = stats_by_id.get(feature_id)
        history[feature_id].append(
            stat.distribution_deviation if stat is not None else None
        )
    latest_stats = stats_by_id

# ドリフト検出結果の表示
print("最新Jobのドリフト検出結果")
for feature_id in FEATURE_IDS:
    stat = latest_stats.get(feature_id)
    if stat is None:
        continue
    print(f"  {feature_id}: deviation={stat.distribution_deviation:.4f} "
          f"drift_detected={stat.drift_detected}")

# 特徴量ごとの推移をプロットする
fig, ax = plt.subplots(figsize=(8, 5))
for feature_id in FEATURE_IDS:
    ax.plot(labels, history[feature_id], marker="o", label=feature_id)
ax.axhline(
    DRIFT_THRESHOLD, color="red", linestyle="--",
    label=f"threshold ({DRIFT_THRESHOLD})",
)
ax.set_title("Feature drift (distribution_deviation) over monitoring jobs")
ax.set_xlabel("Feature Monitoring Job")
ax.set_ylabel("distribution_deviation")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6.14 クリーンアップ

ハンズオンが終わったら、不要な課金を避けるためにリソースを削除します。
同等の処理は `clean_up.py` としても用意しています。

In [ ]:
# リソース削除のフラグ
# True に変更してからセルを実行してください
delete_resources = False  # @param {type:"boolean"}

if delete_resources:
    print("リソースを削除します...\n")

    # 1. FeatureView の削除（force=True により関連する同期ジョブも削除）
    # 継続的同期のシミュレーション（任意セル）を実行していなくても動くよう、IDは直接指定
    for fv_id in [FEATURE_VIEW_REGISTRY_ID, "zone_stats_continuous_view"]:
        try:
            fv_to_delete = feature_store.FeatureView(
                fv_id, feature_online_store_id=FEATURE_ONLINE_STORE_ID
            )
            try:
                fv_to_delete.delete(force=True)
            except TypeError:
                # force 未対応のSDKバージョンでは通常削除
                fv_to_delete.delete()
            print(f"  FeatureView 削除完了: {fv_id}")
        except Exception as e:
            print(f"  FeatureView 削除スキップ ({fv_id}): {e}")

    # 2. FeatureOnlineStore の削除（force=True により残存するFeatureViewも一括削除）
    try:
        fos_to_delete = feature_store.FeatureOnlineStore(FEATURE_ONLINE_STORE_ID)
        fos_to_delete.delete(force=True)
        print(f"  FeatureOnlineStore 削除完了: {FEATURE_ONLINE_STORE_ID}")
    except Exception as e:
        print(f"  FeatureOnlineStore 削除スキップ: {e}")

    # 3. FeatureGroup の削除（force=True により配下のFeature、FeatureMonitorも一括削除）
    try:
        fg_to_delete = feature_store.FeatureGroup(FEATURE_GROUP_ID)
        fg_to_delete.delete(force=True)
        print(f"  FeatureGroup 削除完了: {FEATURE_GROUP_ID}")
    except Exception as e:
        print(f"  FeatureGroup 削除スキップ: {e}")

    # 4. BigQuery データセットの削除（delete_contents=True によりテーブルも一括削除）
    try:
        bq_client.delete_dataset(
            f"{PROJECT_ID}.{BQ_DATASET_ID}",
            delete_contents=True,
            not_found_ok=True,
        )
        print(f"  BigQuery データセット削除完了: {BQ_DATASET_ID}")
    except Exception as e:
        print(f"  BigQuery 削除スキップ: {e}")

    print("\nクリーンアップが完了しました")
else:
    print("delete_resources = False のため、リソースは削除されません。")
    print("削除する場合は delete_resources = True に変更して再実行してください。")